# NLP Sentiment Analysis with VADER

This notebook performs NLP-based sentiment analysis using VADER. Unlike the rule-based notebook, this workflow converts each event record into natural-language text and lets an NLP sentiment analyzer score that text.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.io as pio
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

pio.templates.default = "plotly_dark"

In [2]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "Script":
    PROJECT_ROOT = PROJECT_ROOT.parent

SOURCE_PATH = PROJECT_ROOT / "Data" / "EventManagmentDataSet.csv"
OUTPUT_PATH = PROJECT_ROOT / "Data" / "event_nlp_sentiment_analysis.csv"

event_data = pd.read_csv(SOURCE_PATH)
event_data.head()

,SNO,Event,Faculty_id,Faculty_name,Faculty_Designation,Registers,Type,Request for Booking,Request for Cancelling,Requested Room No,Request Date,Requested Time,Duration,Room Available/Not Available,Accept/Reject
0,1,make a creative logo,FAC010102,John Doe,Associate Professor,100,Important,1,0,179,9/27/2022,3:00pm,120min,Available,Accept
1,2,Presentiong ideas on MSME,FAC010070,Jane Smith,Associate Professor,105,Very important,1,0,323,10/18/2022,3:00pm,120min,Available,Accept
2,3,Webinar on Web-Technologies,FAC010103,Alice Johnson,Associate Professor,77,Normal,1,0,279,10/25/2022,1:00pm,180min,Not Available,Reject
3,4,Group Discussion,FAC010077,Bob Brown,Associate Professor,60,Important,1,0,202,11/1/2022,2:00pm,90min,Available,Accept
4,5,Session on SIH,FAC010060,Carol Davis,Associate Professor,111,Very important,1,0,202,11/8/2022,3:00pm,90min,Available,Accept


## Build NLP Text

VADER works on text, so each structured row is transformed into a short natural-language description of the event request.

In [3]:
def build_event_text(row: pd.Series) -> str:
    event = row["Event"]
    priority = str(row["Type"]).strip()
    registrations = row["Registers"]
    availability = row["Room Available/Not Available"]
    decision = row["Accept/Reject"]
    booking = "booking was requested" if row["Request for Booking"] else "booking was not requested"
    cancellation = "cancellation was requested" if row["Request for Cancelling"] else "cancellation was not requested"

    return (
        f"The event request is for {event}. "
        f"The event priority is {priority}. "
        f"The event has {registrations} registrations. "
        f"The room is {availability}. "
        f"The request decision is {decision}. "
        f"The {booking}. "
        f"The {cancellation}."
    )


event_data["NLP Text"] = event_data.apply(build_event_text, axis=1)
event_data[["Event", "NLP Text"]].head()

,Event,NLP Text
0,make a creative logo,The event request is for make a creative logo....
1,Presentiong ideas on MSME,The event request is for Presentiong ideas on ...
2,Webinar on Web-Technologies,The event request is for Webinar on Web-Techno...
3,Group Discussion,The event request is for Group Discussion. The...
4,Session on SIH,The event request is for Session on SIH. The e...


## Run VADER Sentiment

VADER returns a compound sentiment value from `-1` to `1`. This notebook scales it to `-100` to `100` for consistency with the earlier scoring work.

In [4]:
analyzer = SentimentIntensityAnalyzer()


def sentiment_label(score: float) -> str:
    if score >= 20:
        return "Positive"
    if score <= -20:
        return "Negative"
    return "Neutral"


def clause_reason(row: pd.Series, label: str) -> str:
    clauses = [part.strip() for part in row["NLP Text"].split(".") if part.strip()]
    scored_clauses = []
    for clause in clauses:
        compound = analyzer.polarity_scores(clause)["compound"]
        scored_clauses.append((clause, compound))

    if label == "Positive":
        selected = [clause for clause, score in scored_clauses if score > 0]
        reason_start = "Positive because the NLP model found positive language in"
    elif label == "Negative":
        selected = [clause for clause, score in scored_clauses if score < 0]
        reason_start = "Negative because the NLP model found negative language in"
    else:
        selected = [clause for clause, score in scored_clauses if abs(score) < 0.2]
        reason_start = "Neutral because the NLP model found mostly balanced or weak sentiment in"

    if not selected:
        selected = [max(scored_clauses, key=lambda item: abs(item[1]))[0]]

    return f"{reason_start}: " + "; ".join(selected[:3])


def analyze_nlp_sentiment(row: pd.Series) -> pd.Series:
    scores = analyzer.polarity_scores(row["NLP Text"])
    score = round(scores["compound"] * 100, 2)
    label = sentiment_label(score)

    return pd.Series(
        {
            "NLP Negative": scores["neg"],
            "NLP Neutral": scores["neu"],
            "NLP Positive": scores["pos"],
            "NLP Compound": scores["compound"],
            "NLP Sentiment Score": score,
            "NLP Sentiment Label": label,
            "NLP Sentiment Reason": clause_reason(row, label),
        }
    )


nlp_columns = event_data.apply(analyze_nlp_sentiment, axis=1)
nlp_sentiment_data = pd.concat([event_data, nlp_columns], axis=1)

nlp_sentiment_data[
    [
        "Event",
        "NLP Sentiment Score",
        "NLP Sentiment Label",
        "NLP Sentiment Reason",
    ]
]

,Event,NLP Sentiment Score,NLP Sentiment Label,NLP Sentiment Reason
0,make a creative logo,74.30,Positive,Positive because the NLP model found positive ...
1,Presentiong ideas on MSME,57.09,Positive,Positive because the NLP model found positive ...
2,Webinar on Web-Technologies,-40.19,Negative,Negative because the NLP model found negative ...
3,Group Discussion,52.67,Positive,Positive because the NLP model found positive ...
4,Session on SIH,57.09,Positive,Positive because the NLP model found positive ...
5,Quiz-CODE KNACK WITH C,57.09,Positive,Positive because the NLP model found positive ...
6,JAM,38.18,Positive,Positive because the NLP model found positive ...
7,QUIZ,52.67,Positive,Positive because the NLP model found positive ...
8,WATCHING SITCOM,38.18,Positive,Positive because the NLP model found positive ...
9,Debate,57.09,Positive,Positive because the NLP model found positive ...


In [5]:
nlp_sentiment_data["NLP Sentiment Label"].value_counts()

NLP Sentiment Label
Positive    12
Negative     2
Neutral      1
Name: count, dtype: int64

## NLP Sentiment Visualizations

In [6]:
nlp_counts = (
    nlp_sentiment_data["NLP Sentiment Label"]
    .value_counts()
    .rename_axis("NLP Sentiment Label")
    .reset_index(name="Number of Events")
)

fig = px.bar(
    nlp_counts,
    x="NLP Sentiment Label",
    y="Number of Events",
    color="NLP Sentiment Label",
    text="Number of Events",
    color_discrete_map={"Positive": "#2E7D32", "Neutral": "#757575", "Negative": "#C62828"},
    title="NLP Sentiment Label Distribution",
)
fig.update_traces(textposition="outside")
fig.update_layout(template="plotly_dark", showlegend=False, paper_bgcolor="#111827", plot_bgcolor="#111827")
fig.show()

In [7]:
score_by_event = nlp_sentiment_data.sort_values("NLP Sentiment Score")

fig = px.bar(
    score_by_event,
    x="NLP Sentiment Score",
    y="Event",
    color="NLP Sentiment Label",
    orientation="h",
    hover_data=["Type", "Registers", "Room Available/Not Available", "Accept/Reject", "NLP Sentiment Reason"],
    color_discrete_map={"Positive": "#2E7D32", "Neutral": "#757575", "Negative": "#C62828"},
    title="NLP Sentiment Score by Event",
)
fig.add_vline(x=0, line_width=1, line_dash="dash", line_color="#E5E7EB")
fig.update_layout(template="plotly_dark", xaxis_range=[-100, 100], yaxis_title="", height=650, paper_bgcolor="#111827", plot_bgcolor="#111827")
fig.show()

In [8]:
fig = px.scatter(
    nlp_sentiment_data,
    x="Registers",
    y="NLP Sentiment Score",
    color="NLP Sentiment Label",
    hover_name="Event",
    hover_data=["Type", "Room Available/Not Available", "Accept/Reject", "NLP Sentiment Reason"],
    color_discrete_map={"Positive": "#2E7D32", "Neutral": "#757575", "Negative": "#C62828"},
    title="NLP Sentiment Score vs. Registration Count",
)
fig.add_hline(y=0, line_width=1, line_dash="dash", line_color="#E5E7EB")
fig.update_layout(template="plotly_dark", yaxis_range=[-100, 100], paper_bgcolor="#111827", plot_bgcolor="#111827")
fig.show()

In [9]:
nlp_sentiment_data.to_csv(OUTPUT_PATH, index=False)
OUTPUT_PATH

PosixPath('/Users/hosnieh/Hosnieh_Folders/Projects/Sentiment_Analysis/Data/event_nlp_sentiment_analysis.csv')